[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 模块 2：PyTorch 张量与自动微分

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=103)


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import numpy as np

In [ ]:
torch.__version__

张量既用来编码要处理的信号，也用来表示模型的内部状态和参数。

**通过这种受限的结构操作数据，可以充分发挥 CPU 和 GPU 的性能。**

构造一个 3x5 的未初始化矩阵：


In [ ]:
x = torch.empty(3,5)
print(x.dtype)
print(x)

如果遇到报错，这个 [stackoverflow 链接](https://stackoverflow.com/questions/50617917/overflow-when-unpacking-long-pytorch) 可能有用……


In [ ]:
x = torch.randn(3,5)
print(x)

In [ ]:
print(x.size())

torch.Size 实际上是一个 [tuple](https://docs.python.org/3/tutorial/datastructures.html#tuples-and-sequences)，所以它支持同样的操作。

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=272)


In [ ]:
x.size()[1]

In [ ]:
x.size() == (3,5)

### 与 numpy 的桥梁

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=325)


In [ ]:
y = x.numpy()
print(y)

In [ ]:
a = np.ones(5)
b = torch.from_numpy(a)
print(a.dtype)
print(b)

In [ ]:
c = b.long()
print(c.dtype, c)
print(b.dtype, b)

In [ ]:
xr = torch.randn(3, 5)
print(xr.dtype, xr)

In [ ]:
resb = xr + b
resb

In [ ]:
resc = xr + c
resc

小心类型！


In [ ]:
resb == resc

In [ ]:
torch.set_printoptions(precision=10)

In [ ]:
resb[0,1]

In [ ]:
resc[0,1]

In [ ]:
resc[0,1].dtype

In [ ]:
xr[0,1]

In [ ]:
torch.set_printoptions(precision=4)

### [广播（Broadcasting）](https://numpy.org/doc/stable/user/basics.broadcasting.html)

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=670)

广播会自动扩展维度：在需要运算时，通过复制元素来补齐形状。

1. 如果两个张量中有一个维度数较少，就在它前面补上若干大小为 1 的维度，补到与另一个张量维度数相同；然后
2. 对每一处不匹配的维度，如果两个张量中有一个在该维度大小为 1，就沿这个轴复制元素把它扩展。

如果某一维度上两个张量大小都不匹配、且都不为 1，运算就会失败。


In [ ]:
A = torch.tensor([[1.], [2.], [3.], [4.]])
print(A.size())
B = torch.tensor([[5., -5., 5., -5., 5.]])
print(B.size())
C = A + B

In [ ]:
C

原来的（列）向量
\begin{eqnarray*}
A = \left( \begin{array}{c}
1\\
2\\
3\\
4\\
\end{array}\right)
\end{eqnarray*}
变换成矩阵
\begin{eqnarray*}
A = \left( \begin{array}{ccccc}
1&1&1&1&1\\
2&2&2&2&2\\
3&3&3&3&3\\
4&4&4&4&4
\end{array}\right)
\end{eqnarray*}
原来的（行）向量
\begin{eqnarray*}
B = (5,-5,5,-5,5)
\end{eqnarray*}
变换成矩阵
\begin{eqnarray*}
B = \left( \begin{array}{ccccc}
5&-5&5&-5&5\\
5&-5&5&-5&5\\
5&-5&5&-5&5\\
5&-5&5&-5&5
\end{array}\right)
\end{eqnarray*}
把这两个矩阵相加就得到：
\begin{eqnarray*}
A+B = \left( \begin{array}{ccccc}
6&-4&6&-4&6\\
7&-3&7&-3&7\\
8&-2&8&-2&8\\
9&-1&9&-1&9
\end{array}\right)
\end{eqnarray*}


### 原地修改

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=875)


In [ ]:
x

In [ ]:
xr

In [ ]:
print(x+xr)

In [ ]:
x.add_(xr)
print(x)

任何原地修改张量的操作都会在函数名后加一个 `_`。

例如：`x.fill_(y)`、`x.t_()` 都会改变 `x`。


In [ ]:
print(x.t())

In [ ]:
x.t_()
print(x)

### 共享内存

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=990)

也要小心：修改 torch 张量会改变 numpy 数组，反之亦然……

这在 PyTorch 文档[这里](https://pytorch.org/docs/stable/torch.html#torch.from_numpy)有解释：
`torch.from_numpy` 返回的张量和 ndarray 共享同一块内存。对张量的修改会反映到 ndarray 上，反之亦然。


In [ ]:
a = np.ones(5)
b = torch.from_numpy(a)
print(b)

In [ ]:
a[2] = 0
print(b)

In [ ]:
b[3] = 5
print(a)

### Cuda

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=1120)


In [ ]:
torch.cuda.is_available()

In [ ]:
#device = torch.device('cpu')
device = torch.device('cuda') # 取消注释即可在 GPU 上运行

In [ ]:
x.device

In [ ]:
# 只有 CUDA 可用时才运行这个单元格
# 我们会用 ``torch.device`` 对象把张量移入和移出 GPU
if torch.cuda.is_available():
    y = torch.ones_like(x, device=device)  # 直接在 GPU 上创建张量
    x = x.to(device)                       # 或者直接用字符串 ``.to("cuda")``
    z = x + y
    print(z,z.type())
    print(z.to("cpu", torch.double))       # ``.to`` 还可以同时改变 dtype！

In [ ]:
x = torch.randn(1)
x = x.to(device)

In [ ]:
x.device

In [ ]:
# 下面这一行只有 CUDA 可用时才有用
x = x.data
print(x)
print(x.item())
print(x.cpu().numpy())

# 标准图像数据库的简单接口

[视频时间戳](https://youtu.be/BmAS8IH7n3c?t=1354)

一个例子：[CIFAR10](https://pytorch.org/docs/stable/torchvision/datasets.html#torchvision.datasets.CIFAR10) 数据集。


In [ ]:
import torchvision

data_dir = 'content/data'

cifar = torchvision.datasets.CIFAR10(data_dir, train = True, download = True)
cifar.data.shape

[`permute`](https://pytorch.org/docs/stable/tensors.html#torch.Tensor.permute) 操作的文档。


In [ ]:
x = torch.from_numpy(cifar.data).permute(0,3,1,2).float()
x = x / 255
print(x.type(), x.size(), x.min().item(), x.max().item())

[`narrow(input, dim, start, length)`](https://pytorch.org/docs/stable/torch.html#torch.narrow) 操作的文档。


In [ ]:
# 截取前几张图片并转换为 float
x = torch.narrow(x, 0, 0, 48)

In [ ]:
x.shape

In [ ]:
# 显示图片
def show(img):
    npimg = img.numpy()
    plt.figure(figsize=(20,10))
    plt.imshow(np.transpose(npimg, (1,2,0)), interpolation='nearest')
    
show(torchvision.utils.make_grid(x, nrow = 12))

In [ ]:
# 去掉绿色和蓝色通道
x.narrow(1, 1, 2).fill_(0)
show(torchvision.utils.make_grid(x, nrow = 12))

# Autograd：自动微分

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=40)

在执行张量运算时，PyTorch 可以自动地、即时地构建运算图，用来计算任意量对其中任意张量的梯度。

为了更具体，我们引入下面的例子：考虑参数 $w\in \mathbb{R}$ 和 $b\in \mathbb{R}$ 以及相应的函数：
\begin{eqnarray*}
\ell = \left(\exp(wx+b) - y^* \right)^2
\end{eqnarray*}

我们的目标是计算下面的偏导数：
\begin{eqnarray*}
\frac{\partial \ell}{\partial w}\mbox{ 和 }\frac{\partial \ell}{\partial b}.
\end{eqnarray*}

这样做的原因，等你做这节课的实操练习时就会明白！

你可以把这个函数拆解成基本运算的组合，这就是运算图上的前向传播（forward pass）。
![backprop1](https://dataflowr.github.io/notebooks/Module2/img/backprop1.png)


假设我们先在 `numpy` 里建立模型：


In [ ]:
w = np.array([0.5])
b = np.array([2])
xx = np.array([0.5])#np.arange(0,1.5,.5)

把它们转换成 `tensor`：


In [ ]:
xx_t = torch.from_numpy(xx)
w_t = torch.from_numpy(w)
b_t = torch.from_numpy(b)

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=224)

`tensor` 有一个布尔字段 `requires_grad`，默认为 `False`，它决定 PyTorch 是否要构建运算图，以便计算对它的梯度。


In [ ]:
w_t.requires_grad

我们想对 $w$ 求导，所以改一下这个值：


In [ ]:
w_t.requires_grad_(True)

我们想对 $b$ 做同样的事，但下面这一行会报错！


In [ ]:
b_t.requires_grad_(True)

读懂报错信息应该能帮你改正错误！


In [ ]:
dtype = torch.float64

In [ ]:
b_t = b_t.type(dtype)

In [ ]:
b_t.requires_grad_(True)

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=404)

现在我们计算这个函数：


In [ ]:
def fun(x,ystar):
    y = torch.exp(w_t*x+b_t)
    print(y)
    return torch.sum((y-ystar)**2)

ystar_t = torch.randn_like(xx_t)
l_t = fun(xx_t,ystar_t)

In [ ]:
l_t

In [ ]:
l_t.requires_grad

计算完成之后，也就是*前向传播*结束，你可以调用 ```.backward()```，所有梯度都会自动算好。


In [ ]:
print(w_t.grad)

In [ ]:
l_t.backward()

In [ ]:
print(w_t.grad)
print(b_t.grad)

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=545)

我们来试着理解这些数字……

![backprop2](https://dataflowr.github.io/notebooks/Module2/img/backprop2.png)


In [ ]:
yy_t = torch.exp(w_t*xx_t+b_t)
print(torch.sum(2*(yy_t-ystar_t)*yy_t*xx_t))
print(torch.sum(2*(yy_t-ystar_t)*yy_t))

`tensor.backward()` 会把梯度累加到张量的 `grad` 字段里。


In [ ]:
l_t = fun(xx_t,ystar_t)
l_t.backward()

In [ ]:
print(w_t.grad)
print(b_t.grad)

默认情况下，`backward` 使用后会删除计算图，所以下面你会看到一个报错：


In [ ]:
l_t.backward()

In [ ]:
# 手动把梯度清零
w_t.grad.data.zero_()
b_t.grad.data.zero_()
l_t = fun(xx_t,ystar_t)
l_t.backward(retain_graph=True)
l_t.backward()
print(w_t.grad)
print(b_t.grad)

梯度必须手动清零。否则它们会在多次 _.backward()_ 调用之间不断累加。
这种累加行为在有些场合是我们想要的，比如计算一个在多个"小批次"上累加的损失的梯度，或者多个损失之和的梯度。


[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)